In [ ]:
from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "libs").is_dir()
        ):
            return candidate

    raise RuntimeError("Project root could not be found.")

ROOT = find_project_root()

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

REPORTS_DIR = RESULTS_DIR / "reports"
CURVES_DIR = RESULTS_DIR / "curves_data"
AGGREGATED_REPORTS_DIR = RESULTS_DIR / "aggregated_reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATED_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import multiprocessing
import gc
from scipy.stats import norm
from joblib import Parallel, delayed
import multiprocessing
import concurrent.futures


# ML and Metrics
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, f1_score, roc_auc_score, 
                             precision_score, matthews_corrcoef, precision_recall_curve, 
                             auc, average_precision_score)

# Deep Learning (Attacks)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# from cleverhans.tf2.attacks.carlini_wagner_l2 import carlini_wagner_l2
from libs.carlini_wagner_l2 import carlini_wagner_l2
from tqdm import tqdm
# from art.estimators.classification import KerasClassifier
# from art.attacks.evasion import HopSkipJump

# Official Standard WiSARD Library
import wisardpkg

# ==========================================
# EXPERIMENT CONTROL PANEL (STANDARD WISARD)
# ==========================================
# DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS', 'Edge-IIoT', 'ToN-IoT']
DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS']
RUN_BINARY = True          
RUN_MULTICLASS = False      

ENCODING_TYPES = ['linear', 'gaussian', 'distributive'] 

# ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2', 'C&W', 'HSJA', 'BPDA']
ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2', 'C&W']
EPSILON_LINF = 0.3   
EPSILON_L2 = 3.0     
 

N_JOBS = multiprocessing.cpu_count()
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True) 

print(f"Selected Datasets: {DATASETS_TO_RUN}")
print(f"Enabled Modes: Binary={RUN_BINARY} | Multiclass={RUN_MULTICLASS}")
print(f"Enabled Attacks: {ATTACKS_TO_RUN}")
print(f"Binarizations: {ENCODING_TYPES}")

In [ ]:
# ==========================================
# MODELING AND METRICS FUNCTIONS
# ==========================================
def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model

# >>> NEW: Support for distributive processing <<<
def process_data_vectorized_sequential(data, resolution, enc_type, custom_thresholds=None, chunk_size=20000):
    n_samples, n_features = data.shape
    result = np.empty((n_samples, n_features * resolution), dtype=np.int8)
    
    if enc_type == 'linear':
        indices = np.arange(resolution, dtype=np.int8)
        
    total_chunks = (n_samples + chunk_size - 1) // chunk_size
    
    for i in range(total_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, n_samples)
        chunk = data[start:end]
        
        if enc_type in ['gaussian', 'distributive']:
            # Both use custom_thresholds calculated outside the loop
            bits = (chunk[:, :, None] >= custom_thresholds[None, :, :]).astype(np.int8)
        elif enc_type == 'linear':
            chunk_clipped = np.clip(chunk, 0.0, 1.0)
            limits = (chunk_clipped * resolution).astype(np.int8)
            bits = (limits[:, :, None] > indices[None, None, :]).astype(np.int8)
            
        result[start:end] = bits.reshape(chunk.shape[0], -1)
    return result

def true_parallel_evaluate_worker(addr, X_train_np, y_train_str, X_clean_chunk_np, adv_chunks_dict_np):
    # .tolist() is called HERE INSIDE, already isolated in the C++ core.
    # Joblib uses Memmap to transfer NumPy arrays instantly.
    
    # 1. Train an isolated copy of the network ONLY in this core
    t0 = time.time()
    model = wisardpkg.Wisard(addr, bleachingActivated=True)
    model.train(X_train_np.tolist(), y_train_str)
    train_time = time.time() - t0
    
    # 2. Classify the clean-data chunk (with exhaustive Bleaching)
    t1 = time.time()
    res_clean = model.classify(X_clean_chunk_np.tolist())
    infer_time_clean = time.time() - t1
    
    # 3. Classify the attack chunks
    res_adv = {}
    infer_time_adv = {}
    for atk_name, chunk_np in adv_chunks_dict_np.items():
        t2 = time.time()
        res_adv[atk_name] = model.classify(chunk_np.tolist())
        infer_time_adv[atk_name] = time.time() - t2
        
    return res_clean, res_adv, train_time, infer_time_adv

def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}
    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0
        
    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None
    
    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)
    metrics[f'{context_name}_Precision'] = precision_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_Recall'] = recall_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_F1'] = f1_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_MCC'] = matthews_corrcoef(y_true, y_pred)
    
    mask_normal = (y_true == normal_idx)
    mask_attack = (y_true != normal_idx)
    
    if np.sum(mask_normal) > 0:
        metrics[f'{context_name}_FAR'] = np.sum((y_pred != normal_idx) & mask_normal) / np.sum(mask_normal)
    else:
        metrics[f'{context_name}_FAR'] = 0.0

    if np.sum(mask_attack) > 0:
        metrics[f'{context_name}_ASR'] = np.sum((y_pred == normal_idx) & mask_attack) / np.sum(mask_attack)
    else:
        metrics[f'{context_name}_ASR'] = 0.0

    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)
            prec, rec, _ = precision_recall_curve(y_true, prob_positive)
            metrics[f'{context_name}_PR_AUC'] = auc(rec, prec)
        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
            y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
            metrics[f'{context_name}_PR_AUC'] = average_precision_score(y_true_bin, y_prob, average="macro")
    except Exception as e:
        metrics[f'{context_name}_AUC'] = 0.0
        metrics[f'{context_name}_PR_AUC'] = 0.0
    
    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    for idx, name in enumerate(class_names):
        if idx == normal_idx: continue
        mask_t = (y_true == idx)
        if np.sum(mask_t) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum((y_pred == normal_idx) & mask_t) / np.sum(mask_t)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0
            
    return metrics

In [ ]:
# ==========================================
# MAIN EXPERIMENT LOOP (STANDARD WISARD)
# ==========================================
for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*50}")
    print(f">>> STARTING EXPERIMENTS: {dataset_name.upper()}")
    print(f"{'='*50}")
    
    # ---> CLEANUP BLOCK COMMENTED OUT TO PROTECT OLD CSV FILES <---
    # for f in os.listdir(REPORTS_DIR):
    #     if f.startswith(f'standart_wisard_{dataset_name}'):
    #         os.remove(os.path.join(REPORTS_DIR, f))
            
    # 1. DATA LOADING AND PREPROCESSING
    if dataset_name == 'Bot-IoT':
        df_train = pd.read_csv(DATA_DIR / "botiot" / "BotIoT_training-set.csv")
        df_test = pd.read_csv(DATA_DIR / "botiot" / "BotIoT_testing-set.csv")
    elif dataset_name == 'UNSW-NB15':
        df_train = pd.read_csv(DATA_DIR / "unsw_nb15" / "UNSW_NB15_training-set.csv")
        df_test = pd.read_csv(DATA_DIR / "unsw_nb15" / "UNSW_NB15_testing-set.csv")
    elif dataset_name == 'CICIDS':
        df_train = pd.read_csv(DATA_DIR / "cicids2017" / "CICIDS_training-set.csv")
        df_test = pd.read_csv(DATA_DIR / "cicids2017" / "CICIDS_testing-set.csv")
    # elif dataset_name == 'Edge-IIoT':
    #     df_train = pd.read_csv("data4/Edge-IIoT_training-set.csv")
    #     df_test = pd.read_csv("data4/Edge-IIoT_testing-set.csv")
    # elif dataset_name == 'ToN-IoT':
    #     df_train = pd.read_csv("data5/ToN-IoT_training-set.csv")
    #     df_test = pd.read_csv("data5/ToN-IoT_testing-set.csv")
        
    for df in [df_train, df_test]:
        if 'id' in df.columns: df.drop(columns=['id'], inplace=True)

    y_train_bin = df_train['label'].values
    y_test_bin = df_test['label'].values
    class_names_bin = ['Normal', 'Attack']

    df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
    df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))
    y_train_multi = le.transform(df_train['attack_cat'])
    y_test_multi = le.transform(df_test['attack_cat'])
    class_names_multi = le.classes_

    df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
    df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

    # 1. Find categorical and numerical columns
    categorical_cols = df_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

    # 2. Full safeguard: fill hidden null values before conversion
    for col in categorical_cols:
        df_train[col] = df_train[col].fillna('unknown').astype(str)
        df_test[col] = df_test[col].fillna('unknown').astype(str)

    for col in numerical_cols:
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce').fillna(0.0)
        df_test[col] = pd.to_numeric(df_test[col], errors='coerce').fillna(0.0)

    # 3. Create the preprocessor
    preprocessor = ColumnTransformer([
        ('num', MinMaxScaler(feature_range=(0,1)), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

    print(">>> Applying Scaling and One-Hot Encoding...")
    preprocessor.fit(df_train)
    X_train = preprocessor.transform(df_train).astype('float32')
    X_test = preprocessor.transform(df_test).astype('float32')

    del df_train, df_test
    gc.collect()

    # 2. GENERATION OF ADVERSARIAL ATTACKS AND NOISE
    attacks_dict_bin = {}
    attacks_dict_multi = {}

    if 'FGSM' in ATTACKS_TO_RUN:
        print(">>> Generating FGSM Attack (Transferred White-Box)...")
        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['FGSM'] = fast_gradient_method(logits_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bin, logits_bin
        
        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['FGSM'] = fast_gradient_method(logits_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_multi, logits_multi
        gc.collect()

    if 'RANDOM_LINF' in ATTACKS_TO_RUN:
        print(f">>> Generating Random L-infinity Noise (Eps={EPSILON_LINF})...")
        noise = np.random.uniform(-EPSILON_LINF, EPSILON_LINF, X_test.shape).astype('float32')
        if RUN_BINARY: attacks_dict_bin['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    if 'RANDOM_L2' in ATTACKS_TO_RUN:
        print(f">>> Generating Random L2 Noise (Eps={EPSILON_L2})...")
        noise = np.random.normal(0, 1, X_test.shape).astype('float32')
        norms = np.linalg.norm(noise, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        noise = noise * (EPSILON_L2 / norms)
        if RUN_BINARY: attacks_dict_bin['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    # >>> NEW: C&W ATTACK BLOCK (OPTIMIZED FOR PRESAL-WS20) <<<
    if 'C&W' in ATTACKS_TO_RUN:
        print(">>> Generating C&W L2 Attack (Warning: heavy optimization, processing in large batches)...")
        
        def generate_cw_in_batches(logits_model, X_np, n_classes, batch_size=2500):
            adv_x = []
            total_batches = (len(X_np) + batch_size - 1) // batch_size
            
            for i in tqdm(range(total_batches), desc="Generating C&W Batches", unit="batch"):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, len(X_np))
                
                # Conversion and hard clipping of bounds (0 to 1)
                x_batch = tf.convert_to_tensor(X_np[start_idx:end_idx], dtype=tf.float32)
                x_batch = tf.clip_by_value(x_batch, clip_value_min=0.0, clip_value_max=1.0)
                
                # Dynamic Class Calculation
                preds = logits_model(x_batch)
                y_one_hot = tf.one_hot(tf.argmax(preds, axis=1), depth=n_classes)
                
                adv_batch = carlini_wagner_l2(
                    logits_model, 
                    x_batch, 
                    y=y_one_hot,
                    batch_size=x_batch.shape[0], 
                    clip_min=0.0, 
                    clip_max=1.0,
                    max_iterations=100 
                )
                
                adv_x.append(adv_batch)
                
            return np.vstack(adv_x)

        if RUN_BINARY:
            print("   -> Training base model for C&W (Binary)...")
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['C&W'] = generate_cw_in_batches(logits_bin, X_test, n_classes=2)
            del mlp_bin, logits_bin
        
        if RUN_MULTICLASS:
            print("   -> Training base model for C&W (Multiclass)...")
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['C&W'] = generate_cw_in_batches(logits_multi, X_test, n_classes=len(class_names_multi))
            del mlp_multi, logits_multi

    # if 'HSJA' in ATTACKS_TO_RUN:
    #     print(">>> Generating HopSkipJumpAttack (Decision-Based)...")
    #     if RUN_BINARY:
    #         mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
    #         art_classifier_bin = KerasClassifier(model=mlp_bin, clip_values=(0.0, 1.0), use_logits=False)
    #         # max_iter reduced to save time. Increase it to 50 if you want stronger attacks
    #         hsja = HopSkipJump(classifier=art_classifier_bin, norm=np.inf, max_iter=15, max_eval=1000, init_eval=100)
    #         attacks_dict_bin['HSJA'] = hsja.generate(x=X_test)
    #         del mlp_bin, art_classifier_bin
            
    #     if RUN_MULTICLASS:
    #         mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
    #         art_classifier_multi = KerasClassifier(model=mlp_multi, clip_values=(0.0, 1.0), use_logits=False)
    #         hsja_multi = HopSkipJump(classifier=art_classifier_multi, norm=np.inf, max_iter=15, max_eval=1000, init_eval=100)
    #         attacks_dict_multi['HSJA'] = hsja_multi.generate(x=X_test)
    #         del mlp_multi, art_classifier_multi

    # if 'BPDA' in ATTACKS_TO_RUN:
    #     print(">>> Generating BPDA (Differentiable Approximation of the Backward Pass)...")
        
    #     # 1. Define a binarization with STE (Straight-Through Estimator)
    #     # Bypasses the gradient break of the non-differentiable layer
    #     @tf.custom_gradient
    #     def binarize_ste(x):
    #         forward = tf.cast(x >= 0.5, tf.float32)
    #         def backward(dy):
    #             return dy # The gradient flows as if binarization were the identity function
    #         return forward, backward

    #     # 2. Create an MLP that includes the WiSARD binarization simulator
    #     def build_bpda_mlp(X, y_cat, num_classes):
    #         inputs = Input(shape=(X.shape[1],))
    #         bin_inputs = binarize_ste(inputs) # Forces binarization through BPDA
    #         x = Dense(256, activation='relu')(bin_inputs)
    #         x = Dropout(0.4)(x)
    #         x = Dense(128, activation='relu')(x)
    #         x = Dropout(0.4)(x)
    #         logits = Dense(num_classes, name='logits')(x)
    #         outputs = Activation('softmax')(logits)
    #         model = Model(inputs=inputs, outputs=outputs)
    #         model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    #         model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    #         return model

    #     if RUN_BINARY:
    #         mlp_bpda_bin = build_bpda_mlp(X_train, to_categorical(y_train_bin, 2), 2)
    #         logits_bpda_bin = Model(inputs=mlp_bpda_bin.input, outputs=mlp_bpda_bin.get_layer('logits').output)
    #         attacks_dict_bin['BPDA'] = fast_gradient_method(logits_bpda_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
    #         del mlp_bpda_bin, logits_bpda_bin

    #     if RUN_MULTICLASS:
    #         mlp_bpda_multi = build_bpda_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
    #         logits_bpda_multi = Model(inputs=mlp_bpda_multi.input, outputs=mlp_bpda_multi.get_layer('logits').output)
    #         attacks_dict_multi['BPDA'] = fast_gradient_method(logits_bpda_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
    #         del mlp_bpda_multi, logits_bpda_multi

    gc.collect()

    # 3. STANDARD WISARD HYPERPARAMETERS
    param_grid = {
        'resolution': [1, 2, 4, 8, 10],
        'addressSize': [5, 10, 15, 20]
    }


    # Precomputations for Gaussian and Distributive
    X_mean = X_train.mean(axis=0)
    X_std = X_train.std(axis=0)
    X_std[X_std == 0] = 1e-8 

    # 4. STANDARD WISARD LOOP
    for enc_type in ENCODING_TYPES:
        for res in param_grid['resolution']:
            print(f"\n[{dataset_name} | {enc_type.upper()} | Res={res}] Binarization C...")
            
            custom_thresh = None
            
            # >>> THRESHOLD LOGIC <<<
            if enc_type == 'gaussian':
                skews = [norm.ppf((i+1)/(res+1)) for i in range(res)]
                custom_thresh = X_mean[:, None] + (X_std[:, None] * skews)
            elif enc_type == 'distributive':
                percentiles = np.linspace(0, 100, res + 2)[1:-1]
                custom_thresh = np.percentile(X_train, percentiles, axis=0).T 

            # Binarizing the clean data
            X_train_bin = process_data_vectorized_sequential(X_train, res, enc_type, custom_thresh)
            X_test_bin = process_data_vectorized_sequential(X_test, res, enc_type, custom_thresh)
            
            # Binarizing active attack matrices
            bin_adv_dict = {}
            if RUN_BINARY:
                for atk_name, X_adv_matrix in attacks_dict_bin.items():
                    bin_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)
                    
            multi_adv_dict = {}
            if RUN_MULTICLASS:
                for atk_name, X_adv_matrix in attacks_dict_multi.items():
                    multi_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)

            modes_to_run = []
            if RUN_BINARY: modes_to_run.append('binary')
            if RUN_MULTICLASS: modes_to_run.append('multiclass')

            for mode in modes_to_run:
                if mode == 'binary':
                    y_train_str = [str(y) for y in y_train_bin] 
                    y_test_curr = y_test_bin
                    class_names_curr = class_names_bin
                    active_adv_dict = bin_adv_dict
                else:
                    y_train_str = [str(y) for y in y_train_multi]
                    y_test_curr = y_test_multi
                    class_names_curr = class_names_multi
                    active_adv_dict = multi_adv_dict

                num_classes = len(class_names_curr)
                csv_name = REPORTS_DIR / f"standart_wisard_{dataset_name}_{mode}_{enc_type}.csv"

                for addr in param_grid['addressSize']:
                    print(f"     [Addr={addr} | Mode={mode}] Training and evaluating Standard WiSARD (True Multiprocessing)...")
                    
                    n_jobs = max(1, multiprocessing.cpu_count() - 1)
                    
                    # 1. Split the data into N exact large chunks
                    clean_chunks = np.array_split(X_test_bin, n_jobs)
                    
                    adv_chunks_list = [] 
                    for i in range(n_jobs):
                        chunk_dict = {}
                        for atk_name, X_adv_matrix in active_adv_dict.items():
                            chunk_dict[atk_name] = np.array_split(X_adv_matrix, n_jobs)[i]
                        adv_chunks_list.append(chunk_dict)
                        
                    # 2. Start full parallelism (bypassing the C++ GIL)
                    # We now use 'loky' instead of threads!
                    results = Parallel(n_jobs=n_jobs, backend='loky')(
                        delayed(true_parallel_evaluate_worker)(
                            addr, X_train_bin, y_train_str, clean_chunks[i], adv_chunks_list[i]
                        ) for i in range(n_jobs)
                    )
                    
                    # 3. Merge the results from all 23 workers
                    yp_clean_str = []
                    yp_adv_str_dict = {atk: [] for atk in active_adv_dict.keys()}
                    
                    for res_clean, res_adv, t_train, t_adv_dict in results:
                        yp_clean_str.extend(res_clean)
                        for atk in active_adv_dict.keys():
                            yp_adv_str_dict[atk].extend(res_adv[atk])
                            
                    # --- Metrics Calculation ---
                    yp_clean = np.array([int(y) for y in yp_clean_str])
                    yp_clean_prob = to_categorical(yp_clean, num_classes)
                    m_clean = calculate_miss_rates(y_test_curr, yp_clean, yp_clean_prob, "Clean", class_names_curr)
                    
                    # Since training runs in parallel, we use the training time of the slowest worker (represents real-world time)
                    train_time_real = max([r[2] for r in results])
                    
                    for atk_name, yp_adv_str in yp_adv_str_dict.items():
                        yp_adv = np.array([int(y) for y in yp_adv_str])
                        yp_adv_prob = to_categorical(yp_adv, num_classes)
                        
                        # Total attack inference time is the time of the slowest worker
                        infer_time_adv_real = max([r[3][atk_name] for r in results])
                        
                        m_adv = calculate_miss_rates(y_test_curr, yp_adv, yp_adv_prob, "Adv", class_names_curr)
                        acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']
                        
                        row = {
                            'Dataset': dataset_name,
                            'Attack': atk_name,
                            'Resolution': res, 
                            'AddressSize': addr, 
                            'TrainTime_s': train_time_real, 
                            'InferTime_Adv_s': infer_time_adv_real,    
                            'Acc_Drop_pp': acc_drop*100
                        }
                        row.update(m_clean)
                        row.update(m_adv)
                        
                        row_df = pd.DataFrame([row])
                        file_exists = os.path.exists(csv_name)
                        row_df.to_csv(csv_name, mode='a', header=not file_exists, index=False)
                        
                        file_tag = f"std_wisard_{dataset_name}_{mode}_{enc_type}_{atk_name}_R{res}_A{addr}"
                        np.savez(CURVES_DIR / f"{file_tag}.npz", 
                                model_name=f"Std WiSARD ({enc_type})",
                                attack_name=atk_name,
                                y_true=y_test_curr, 
                                y_prob_clean=yp_clean_prob, 
                                y_prob_adv=yp_adv_prob,
                                class_names=class_names_curr)
                    
                    # Final memory cleanup
                    del results, clean_chunks, adv_chunks_list, yp_clean_str, yp_adv_str_dict
                    gc.collect()
                        

            del X_train_bin, X_test_bin, bin_adv_dict, multi_adv_dict
            gc.collect()

    del attacks_dict_bin, attacks_dict_multi, X_train, X_test
    gc.collect()

print("\n🚀 STANDARD WISARD EXPERIMENT COMPLETED SUCCESSFULLY!")